#  ⭐ `fat_reacoes`


In [1]:
%run ../../config/bootstrap.py

import pandas as pd
import numpy as np
from pathlib import Path
from utils import get_project_root, mapping_column, normalize_date_column, normalize_rows, normalize_duration, expandir_gravidade_wide

In [2]:
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [3]:
path = Path(project_root) / "data/01_bronze/Reacoes/Reacoes.parquet"
bronze = pd.read_parquet(path) 
bronze.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'REACAO_EVTO_ADVERSO_MEDDRA_LLT', 'PT',
       'HLT', 'HLGT', 'SOC', 'DATA_INICIO_HORA', 'DATA_FINAL_HORA', 'DURACAO',
       'GRAVE', 'GRAVIDADE', 'DESFECHO', 'ATUALIZADO'],
      dtype='object')

### Head

In [4]:
bronze.head(2)

,IDENTIFICACAO_NOTIFICACAO,REACAO_EVTO_ADVERSO_MEDDRA_LLT,PT,HLT,HLGT,SOC,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVE,GRAVIDADE,DESFECHO,ATUALIZADO
0,BR-ANVISA-300000004,Coceira,Prurido,Prurido NCO,Quadros clínicos epidérmicos e dérmicos,Distúrbios dos tecidos cutâneos e subcutâneos,None,None,3 dia,Não,None,Recuperado/Resolvido,2025-11-17
1,BR-ANVISA-300000005,Edema periorbital,Edema periorbital,Distúrbios oculares NCO,Transtornos oculares NCO,Distúrbios oculares,20181122,20181122,None,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17


### GRAVE

In [5]:
bronze.GRAVE.value_counts()

GRAVE
Não    374682
Sim    268114
Name: count, dtype: int64

### DESFECHO

In [6]:
bronze.DESFECHO.value_counts()

DESFECHO
Recuperado/Resolvido                         278670
Desconhecido                                 226494
Não Recuperado/Não Resolvido/Em andamento     85241
Em recuperação/Resolvendo                     66908
Fatal/Óbito                                   18581
Recuperado/Resolvido com sequelas              4311
Name: count, dtype: int64

### GRAVIDADE

In [7]:
bronze.GRAVIDADE.value_counts()

GRAVIDADE
Outro efeito clinicamente significativo                                                                                                                                                                              144833
Hospitalização/Prolongamento de hospitalização                                                                                                                                                                        57343
Hospitalização/Prolongamento de hospitalização, Outro efeito clinicamente significativo                                                                                                                               15621
Resultou em óbito                                                                                                                                                                                                     11510
Ameaça à vida                                                                                                 

In [8]:
 
gravidades_unicas = (
    bronze['GRAVIDADE']
    .dropna()
    .astype(str)
    .str.split(r'\s*,\s*')   # separa pelos ","
    .explode()               # uma linha por item
    .str.strip()             # tira espaços extras
    .drop_duplicates()       # remove duplicados
    .sort_values()           # opcional: ordena
)

print(gravidades_unicas.to_list())


['Ameaça à vida', 'Anomalia congênita ou malformação ao nascer', 'Hospitalização/Prolongamento de hospitalização', 'Incapacidade persistente ou significativa', 'Outro efeito clinicamente significativo', 'Resultou em óbito']


In [9]:
for g in gravidades_unicas:
    print(g)

Ameaça à vida
Anomalia congênita ou malformação ao nascer
Hospitalização/Prolongamento de hospitalização
Incapacidade persistente ou significativa
Outro efeito clinicamente significativo
Resultou em óbito


### DURACAO

In [10]:
bronze.DURACAO.value_counts().head(10)

DURACAO
1 dia        27059
2 dia        10017
3 dia         6650
1 hora        6314
30 minuto     5865
4 dia         4278
5 dia         3707
0 dia         3349
2 hora        3112
20 minuto     2698
Name: count, dtype: int64

# 🥈 Silver

normalized data, medium quality


In [11]:
silver = bronze.copy()

### missing

### pk_soc  pk_llt

In [12]:
dim_soc = pd.read_parquet(Path(project_root) / "data/03_gold/dim_soc/dim_soc.parquet")
dim_soc.head()

,PK_SOC,SOC,HLGT,HLT,PT,PK_LLT,REACAO_EVTO_ADVERSO_MEDDRA_LLT
0,26,Circunstâncias sociais,Fatores relacionados ao gênero,Circunstâncias relacionadas à gravidez,Amamentação,1,Amamentação
1,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Abstinência sexual,2,Abstinência sexual
2,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Atividade sexual aumentada,3,Atividade sexual aumentada
3,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Não consumação,4,Não consumação
4,26,Circunstâncias sociais,Fatores relacionados à idade,Questões relacionadas à idade,Bebê,5,Neonato


In [13]:
dim_soc.columns

Index(['PK_SOC', 'SOC', 'HLGT', 'HLT', 'PT', 'PK_LLT',
       'REACAO_EVTO_ADVERSO_MEDDRA_LLT'],
      dtype='object')

In [14]:
hist_fat_reacoes = silver.merge(dim_soc, on=['REACAO_EVTO_ADVERSO_MEDDRA_LLT', 'PT', 'HLT', 'HLGT','SOC'], how='left')
hist_fat_reacoes = hist_fat_reacoes.drop(['SOC', 'HLGT', 'HLT', 'PT','REACAO_EVTO_ADVERSO_MEDDRA_LLT'], axis=1)
hist_fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVE,GRAVIDADE,DESFECHO,ATUALIZADO,PK_SOC,PK_LLT
0,BR-ANVISA-300000004,None,None,3 dia,Não,None,Recuperado/Resolvido,2025-11-17,16,4114
1,BR-ANVISA-300000005,20181122,20181122,None,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17,9,9107
2,BR-ANVISA-300000007,20181115,None,2 dia,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17,16,3903
3,BR-ANVISA-300000008,20181025,None,5 dia,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17,12,11411
4,BR-ANVISA-300000010,201508,201508,None,Sim,Hospitalização/Prolongamento de hospitalização,Recuperado/Resolvido,2025-11-17,8,1994


#### date

In [15]:
col_dates = ["DATA_INICIO_HORA", "DATA_FINAL_HORA"]

for col in col_dates:
    if col in hist_fat_reacoes.columns:
        hist_fat_reacoes[col] = normalize_date_column(hist_fat_reacoes[col])

hist_fat_reacoes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 815877 entries, 0 to 815876
Data columns (total 10 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   IDENTIFICACAO_NOTIFICACAO  815877 non-null  object        
 1   DATA_INICIO_HORA           347696 non-null  datetime64[ns]
 2   DATA_FINAL_HORA            189475 non-null  datetime64[ns]
 3   DURACAO                    127726 non-null  object        
 4   GRAVE                      642796 non-null  object        
 5   GRAVIDADE                  267804 non-null  object        
 6   DESFECHO                   680205 non-null  object        
 7   ATUALIZADO                 815877 non-null  object        
 8   PK_SOC                     770393 non-null  Int64         
 9   PK_LLT                     815877 non-null  int64         
dtypes: Int64(1), datetime64[ns](2), int64(1), object(6)
memory usage: 63.0+ MB


In [16]:
hist_fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVE,GRAVIDADE,DESFECHO,ATUALIZADO,PK_SOC,PK_LLT
0,BR-ANVISA-300000004,NaT,NaT,3 dia,Não,None,Recuperado/Resolvido,2025-11-17,16,4114
1,BR-ANVISA-300000005,2018-11-22,2018-11-22,None,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17,9,9107
2,BR-ANVISA-300000007,2018-11-15,NaT,2 dia,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17,16,3903
3,BR-ANVISA-300000008,2018-10-25,NaT,5 dia,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17,12,11411
4,BR-ANVISA-300000010,NaT,NaT,None,Sim,Hospitalização/Prolongamento de hospitalização,Recuperado/Resolvido,2025-11-17,8,1994


### GRAVE

In [17]:
hist_fat_reacoes.GRAVE.value_counts()

GRAVE
Não    374682
Sim    268114
Name: count, dtype: int64

In [18]:
 
grave_map = {
    "Desconhecido": 0,
    "Sim": 1,
    "Não": 2, 
}
hist_fat_reacoes['GRAVE_TIPO'] = hist_fat_reacoes['GRAVE'].fillna("Desconhecido") 
hist_fat_reacoes['GRAVE_VALOR'] = hist_fat_reacoes['GRAVE'].map(grave_map).astype('Int64').fillna(0)
hist_fat_reacoes = hist_fat_reacoes.drop('GRAVE', axis=1)
hist_fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVIDADE,DESFECHO,ATUALIZADO,PK_SOC,PK_LLT,GRAVE_TIPO,GRAVE_VALOR
0,BR-ANVISA-300000004,NaT,NaT,3 dia,None,Recuperado/Resolvido,2025-11-17,16,4114,Não,2
1,BR-ANVISA-300000005,2018-11-22,2018-11-22,None,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17,9,9107,Sim,1
2,BR-ANVISA-300000007,2018-11-15,NaT,2 dia,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17,16,3903,Sim,1
3,BR-ANVISA-300000008,2018-10-25,NaT,5 dia,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17,12,11411,Sim,1
4,BR-ANVISA-300000010,NaT,NaT,None,Hospitalização/Prolongamento de hospitalização,Recuperado/Resolvido,2025-11-17,8,1994,Sim,1


In [19]:
hist_fat_reacoes[['GRAVE_TIPO', 'GRAVE_VALOR']].drop_duplicates()

,GRAVE_TIPO,GRAVE_VALOR
0,Não,2
1,Sim,1
36,Desconhecido,0


###  DESFECHO

In [20]:
hist_fat_reacoes.DESFECHO.value_counts()

DESFECHO
Recuperado/Resolvido                         278670
Desconhecido                                 226494
Não Recuperado/Não Resolvido/Em andamento     85241
Em recuperação/Resolvendo                     66908
Fatal/Óbito                                   18581
Recuperado/Resolvido com sequelas              4311
Name: count, dtype: int64

In [21]:
desfecho_map = {
    "Desconhecido": 0, 
    "Não Recuperado/Não Resolvido/Em andamento": 1, 
    "Em recuperação/Resolvendo": 2, 
    "Recuperado/Resolvido": 3,
    "Fatal/Óbito": 4, 
    "Recuperado/Resolvido com sequelas": 5, 
}
hist_fat_reacoes['DESFECHO_TIPO'] = hist_fat_reacoes['DESFECHO'].fillna("Desconhecido")  
hist_fat_reacoes['DESFECHO_VALOR'] = hist_fat_reacoes['DESFECHO'].map(desfecho_map).astype('Int64').fillna(0)
hist_fat_reacoes = hist_fat_reacoes.drop('DESFECHO', axis=1)
hist_fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVIDADE,ATUALIZADO,PK_SOC,PK_LLT,GRAVE_TIPO,GRAVE_VALOR,DESFECHO_TIPO,DESFECHO_VALOR
0,BR-ANVISA-300000004,NaT,NaT,3 dia,None,2025-11-17,16,4114,Não,2,Recuperado/Resolvido,3
1,BR-ANVISA-300000005,2018-11-22,2018-11-22,None,Outro efeito clinicamente significativo,2025-11-17,9,9107,Sim,1,Recuperado/Resolvido,3
2,BR-ANVISA-300000007,2018-11-15,NaT,2 dia,Outro efeito clinicamente significativo,2025-11-17,16,3903,Sim,1,Recuperado/Resolvido,3
3,BR-ANVISA-300000008,2018-10-25,NaT,5 dia,Outro efeito clinicamente significativo,2025-11-17,12,11411,Sim,1,Recuperado/Resolvido,3
4,BR-ANVISA-300000010,NaT,NaT,None,Hospitalização/Prolongamento de hospitalização,2025-11-17,8,1994,Sim,1,Recuperado/Resolvido,3


In [22]:
hist_fat_reacoes[['DESFECHO_TIPO', 'DESFECHO_VALOR']].drop_duplicates()

,DESFECHO_TIPO,DESFECHO_VALOR
0,Recuperado/Resolvido,3
5,Desconhecido,0
39,Em recuperação/Resolvendo,2
53,Não Recuperado/Não Resolvido/Em andamento,1
64,Fatal/Óbito,4
331,Recuperado/Resolvido com sequelas,5


### DURACAO

In [23]:
hist_fat_reacoes.DURACAO.value_counts().head(10)

DURACAO
1 dia        27059
2 dia        10017
3 dia         6650
1 hora        6314
30 minuto     5865
4 dia         4278
5 dia         3707
0 dia         3349
2 hora        3112
20 minuto     2698
Name: count, dtype: int64

In [24]:
hist_fat_reacoes['DURACAO'] = hist_fat_reacoes['DURACAO'].replace(["", " ", "NAN", "NONE", "NULL","NaN"], np.nan).fillna("desconhecido")
hist_fat_reacoes = normalize_duration(hist_fat_reacoes, col='DURACAO', prefix='DURACAO')
hist_fat_reacoes['DURACAO_TIPO'] = hist_fat_reacoes['DURACAO_TIPO'].replace(["", " ", "NAN", "NONE", "NULL","NaN"], np.nan).fillna("desconhecido")
#hist_fat_reacoes = hist_fat_reacoes.drop('DURACAO', axis=1)


In [51]:
hist_fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVIDADE,ATUALIZADO,PK_SOC,PK_LLT,GRAVE_TIPO,GRAVE_VALOR,DESFECHO_TIPO,DESFECHO_VALOR,DURACAO_VALOR,DURACAO_TIPO
0,BR-ANVISA-300000004,NaT,NaT,3 dia,None,2025-11-17,16,4114,Não,2,Recuperado/Resolvido,3,3.0,DIA(S)
1,BR-ANVISA-300000005,2018-11-22,2018-11-22,DESCONHECIDO,Outro efeito clinicamente significativo,2025-11-17,9,9107,Sim,1,Recuperado/Resolvido,3,NaN,desconhecido
2,BR-ANVISA-300000007,2018-11-15,NaT,2 dia,Outro efeito clinicamente significativo,2025-11-17,16,3903,Sim,1,Recuperado/Resolvido,3,2.0,DIA(S)
3,BR-ANVISA-300000008,2018-10-25,NaT,5 dia,Outro efeito clinicamente significativo,2025-11-17,12,11411,Sim,1,Recuperado/Resolvido,3,5.0,DIA(S)
4,BR-ANVISA-300000010,NaT,NaT,DESCONHECIDO,Hospitalização/Prolongamento de hospitalização,2025-11-17,8,1994,Sim,1,Recuperado/Resolvido,3,NaN,desconhecido


In [25]:
hist_fat_reacoes[['DURACAO_TIPO']].drop_duplicates()

,DURACAO_TIPO
0,DIA(S)
1,desconhecido
9,MES(ES)
14,HORA(S)
43,MINUTO(S)
63,SEMANA(S)
88,SEGUNDO(S)
158,ANO(S)
331827,DECADA(S)


In [27]:
hist_fat_reacoes.DURACAO_VALOR.value_counts().head(15)

DURACAO_VALOR
1.0     37128
2.0     15479
3.0      9917
30.0     6422
5.0      6190
4.0      6146
10.0     4270
6.0      3670
0.0      3481
20.0     3372
7.0      3269
15.0     2735
8.0      2652
40.0     1979
12.0     1612
Name: count, dtype: int64

### GRAVIDADE

In [28]:
hist_fat_reacoes = expandir_gravidade_wide(hist_fat_reacoes, col='GRAVIDADE', prefix='GRAVIDADE_')
hist_fat_reacoes = hist_fat_reacoes.drop('GRAVIDADE', axis=1)
hist_fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,ATUALIZADO,PK_SOC,PK_LLT,GRAVE_TIPO,GRAVE_VALOR,DESFECHO_TIPO,DESFECHO_VALOR,DURACAO_VALOR,DURACAO_TIPO,GRAVIDADE_RESULTADO_OBITO,GRAVIDADE_AMEACA_VIDA,GRAVIDADE_INCAPACIDADE,GRAVIDADE_HOSPITALIZACAO,GRAVIDADE_OUTRO_EFEITO,GRAVIDADE_ANOMALIA_CONGENITA
0,BR-ANVISA-300000004,NaT,NaT,3 dia,2025-11-17,16,4114,Não,2,Recuperado/Resolvido,3,3.0,DIA(S),0,0,0,0,0,0
1,BR-ANVISA-300000005,2018-11-22,2018-11-22,desconhecido,2025-11-17,9,9107,Sim,1,Recuperado/Resolvido,3,NaN,desconhecido,0,0,0,0,1,0
2,BR-ANVISA-300000007,2018-11-15,NaT,2 dia,2025-11-17,16,3903,Sim,1,Recuperado/Resolvido,3,2.0,DIA(S),0,0,0,0,1,0
3,BR-ANVISA-300000008,2018-10-25,NaT,5 dia,2025-11-17,12,11411,Sim,1,Recuperado/Resolvido,3,5.0,DIA(S),0,0,0,0,1,0
4,BR-ANVISA-300000010,NaT,NaT,desconhecido,2025-11-17,8,1994,Sim,1,Recuperado/Resolvido,3,NaN,desconhecido,0,0,0,1,0,0


### SAVE

In [29]:
hist_fat_reacoes.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'DATA_INICIO_HORA', 'DATA_FINAL_HORA',
       'DURACAO', 'ATUALIZADO', 'PK_SOC', 'PK_LLT', 'GRAVE_TIPO',
       'GRAVE_VALOR', 'DESFECHO_TIPO', 'DESFECHO_VALOR', 'DURACAO_VALOR',
       'DURACAO_TIPO', 'GRAVIDADE_RESULTADO_OBITO', 'GRAVIDADE_AMEACA_VIDA',
       'GRAVIDADE_INCAPACIDADE', 'GRAVIDADE_HOSPITALIZACAO',
       'GRAVIDADE_OUTRO_EFEITO', 'GRAVIDADE_ANOMALIA_CONGENITA'],
      dtype='object')

In [31]:
cols = ['IDENTIFICACAO_NOTIFICACAO', 'DATA_INICIO_HORA', 'DATA_FINAL_HORA',
       #, 'ATUALIZADO', 
       'PK_SOC', 'PK_LLT', 
       'GRAVE_TIPO','GRAVE_VALOR', 
       'DESFECHO_TIPO', 'DESFECHO_VALOR', 
       'DURACAO_TIPO','DURACAO_VALOR',#'DURACAO'
       'GRAVIDADE_RESULTADO_OBITO', 'GRAVIDADE_AMEACA_VIDA',
       'GRAVIDADE_INCAPACIDADE', 'GRAVIDADE_HOSPITALIZACAO',
       'GRAVIDADE_OUTRO_EFEITO', 'GRAVIDADE_ANOMALIA_CONGENITA']

hist_fat_reacoes[cols].to_parquet(Path(project_root) / "data/02_silver/hist_fat_reacoes/hist_fat_reacoes.parquet", index=False)

# 🥇 Gold


In [32]:
fat_reacoes = hist_fat_reacoes.copy()
fat_reacoes.to_parquet(Path(project_root) / "data/03_gold/fat_reacoes/fat_reacoes.parquet", index=False)